In [ ]:
import transformers
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten") #'microsoft/trocr-base-handwritten'
model = VisionEncoderDecoderModel.from_pretrained('./checkpoint_eval_2014_small_stage1_new_image/checkpoint-19000')


In [ ]:
from PIL import Image, ImageEnhance
import cv2, numpy as np

img_path = 'cub3.jpg'  #'./data2/2014/18_em_1.bmp'   './data2/train/2_em_7.bmp'

In [ ]:

img = Image.open(img_path).convert("RGB")
img = ImageEnhance.Sharpness(img).enhance(2.0)
# img = ImageEnhance.Brightness(img).enhance(2)
# img = ImageEnhance.Color(img).enhance(0.0)


# Display or save
img

In [ ]:
pixel_values = processor(img, return_tensors="pt").pixel_values
print(pixel_values)
print(pixel_values.shape)

greedy search

In [ ]:
generated_ids = model.generate(pixel_values)
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Recognized Latex Text:", generated_text)

beam search

In [ ]:
beam_output = model.generate(
    pixel_values, 
    num_beams=10, 
    early_stopping=True,
    num_return_sequences=5,
    max_length = 490
    #no_repeat_ngram_size = 3
)
print(processor.batch_decode(beam_output, skip_special_tokens=True)[0])
print(processor.batch_decode(beam_output, skip_special_tokens=True)[1])
print(processor.batch_decode(beam_output, skip_special_tokens=True)[2])
print(processor.batch_decode(beam_output, skip_special_tokens=True)[3])
print(processor.batch_decode(beam_output, skip_special_tokens=True)[4])


#max_length = 預測的字數
#no_repeat_ngram_size = 0(無窮大) 不出現重複的字幾次

Top-k

In [ ]:
sample_output = model.generate(
    pixel_values, 
    do_sample=True, 
    top_k=50
)
print(processor.batch_decode(sample_output, skip_special_tokens=True)[0])

In [ ]:
def ocr_image(src_img):  #greedy
 pixel_values = processor(images=src_img, return_tensors="pt").pixel_values
 generated_ids = model.generate(pixel_values)
 return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
def ocr_image2(src_img):  #beam search
 pixel_values = processor(images=src_img, return_tensors="pt").pixel_values
 generated_ids = model.generate(pixel_values, num_beams=5,early_stopping=True)
 return processor.batch_decode(generated_ids, skip_special_tokens=False)[0]

In [ ]:
ocr_image(img)

ocr_image2(img)